# Data Wrangling

Cleans the merged dataset: handles missing values, builds the binary landing-outcome label, and one-hot encodes categorical features for modeling.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('spacex_launch_geo.csv')
print("Raw shape:", df.shape)
print(df.isnull().sum())

# --- Handle missing values ---
# Payload Mass: fill missing with column mean
mean_payload = df['Payload Mass (kg)'].mean()
df['Payload Mass (kg)'] = df['Payload Mass (kg)'].fillna(mean_payload)

# --- Create the binary landing-outcome label ---
# class: 1 = successful landing, 0 = failure / no attempt
bad_outcomes = {'None None', 'None ASDS', 'None RTLS', 'False ASDS', 'False Ocean', 'False RTLS'}
df['class'] = df['class'].astype(int)  # already provided as 0/1 in this dataset

# --- Extract a simplified booster-version group for modeling ---
df['BoosterGroup'] = df['Booster Version'].str.extract(r'(F9 v1\.0|F9 v1\.1|F9 FT|F9 B4|F9 B5)')
df['BoosterGroup'] = df['BoosterGroup'].fillna('Other')

# --- One-hot encode categorical features used for modeling ---
features_one_hot = pd.get_dummies(
    df[['Payload Mass (kg)', 'Orbit', 'Launch Site', 'BoosterGroup']],
    columns=['Orbit', 'Launch Site', 'BoosterGroup']
)
features_one_hot = features_one_hot.astype('float64')

print("Wrangled feature matrix shape:", features_one_hot.shape)
print(features_one_hot.head())

# --- Save wrangled outputs for downstream EDA / SQL / ML notebooks ---
df.to_csv('dataset_part_2.csv', index=False)          # cleaned + labeled dataset
features_one_hot.to_csv('dataset_part_3.csv', index=False)  # ML-ready one-hot features
print("Saved dataset_part_2.csv (cleaned) and dataset_part_3.csv (one-hot features)")
